# FLM sampler demo: OpenWebText

This notebook loads the pretrained OpenWebText FLM checkpoint and samples with the standard ODE sampler, the SDE baseline, and the stochastic marginally exact sampler. The last cell evaluates entropy and generative perplexity on 100 samples per sampler.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import torch

from omegaconf import OmegaConf

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import dataloader
from samplers import build_sampler


In [ ]:
checkpoint_candidates = []
if os.environ.get("FLM_CHECKPOINT_PATH"):
    checkpoint_candidates.append(Path(os.environ["FLM_CHECKPOINT_PATH"]).expanduser())
checkpoint_candidates.extend([
    ROOT / "checkpoints" / "flm" / "owt_flm.ckpt",
    ROOT / "checkpoints" / "owt_flm.ckpt",
])

checkpoint_path = next((path for path in checkpoint_candidates if path.exists()), None)
if checkpoint_path is None:
    output_dir = ROOT / "checkpoints" / "flm"
    print(f"Checkpoint not found locally. Downloading into {output_dir} ...")
    subprocess.run(
        [
            sys.executable,
            str(ROOT / "scripts" / "download_flm_checkpoints.py"),
            "--output-dir",
            str(output_dir),
            "--checkpoint",
            "owt_flm.ckpt",
        ],
        check=True,
    )
    checkpoint_path = output_dir / "owt_flm.ckpt"

if not checkpoint_path.exists():
    raise FileNotFoundError(f"Missing checkpoint after download attempt: {checkpoint_path}")

print(f"Using checkpoint: {checkpoint_path}")

config = OmegaConf.load(ROOT / "configs" / "config.yaml")
config.data = OmegaConf.load(ROOT / "configs" / "data" / "openwebtext-split.yaml")
config.model = OmegaConf.load(ROOT / "configs" / "model" / "small.yaml")
config.algo = OmegaConf.load(ROOT / "configs" / "algo" / "flm.yaml")

config.model.length = 1024
config.eval.checkpoint_path = str(checkpoint_path)
config.loader.eval_batch_size = 2
config.sampling.steps = 1024
config.sampling.use_float64 = True

tokenizer = dataloader.get_tokenizer(config)


In [ ]:
num_samples = 2
num_steps = 64
marginally_exact_temperature = 0.9
marginally_exact_p_nucleus = 0.95

def make_sampler(name: str):
    sampler_config = OmegaConf.create(
        OmegaConf.to_container(config, resolve=False)
    )
    sampler_config.sampling.flm_sampler = name
    sampler_config.sampling.temperature = marginally_exact_temperature
    sampler_config.sampling.p_nucleus = marginally_exact_p_nucleus
    sampler_config.sampling.marginally_exact_temperature = marginally_exact_temperature
    sampler_config.sampling.marginally_exact_p_nucleus = marginally_exact_p_nucleus
    return build_sampler(sampler_config, tokenizer)

ode_sampler = make_sampler("ode")
sde_sampler = make_sampler("sde")
marginally_exact_sampler = make_sampler("marginally_exact")


In [ ]:
ode_tokens = ode_sampler.sample(num_samples=num_samples, num_steps=num_steps)
sde_tokens = sde_sampler.sample(num_samples=num_samples, num_steps=num_steps)
marginally_exact_tokens = marginally_exact_sampler.sample(num_samples=num_samples, num_steps=num_steps)

ode_text = tokenizer.batch_decode(ode_tokens, skip_special_tokens=True)
sde_text = tokenizer.batch_decode(sde_tokens, skip_special_tokens=True)
marginally_exact_text = tokenizer.batch_decode(marginally_exact_tokens, skip_special_tokens=True)

print("ODE samples:\n")
for idx, text in enumerate(ode_text):
    print(f"[{idx}] {text[:300]}\n")

print("SDE samples:\n")
for idx, text in enumerate(sde_text):
    print(f"[{idx}] {text[:300]}\n")

print("Marginally exact samples:\n")
for idx, text in enumerate(marginally_exact_text):
    print(f"[{idx}] {text[:300]}\n")


In [ ]:
metric_num_samples = 100

def evaluate_sampler(name, sampler, total_samples=metric_num_samples, num_steps=num_steps):
    sampler.metrics.reset()
    all_tokens = []
    all_texts = []
    remaining = total_samples
    batch_size = int(config.loader.eval_batch_size)
    print(f"Evaluating sampler '{name}' with {total_samples} samples (batch size: {batch_size}) ...")
    while remaining > 0:
        current_batch = min(batch_size, remaining)
        tokens = sampler.sample(num_samples=current_batch, num_steps=num_steps)
        texts = tokenizer.batch_decode(tokens, skip_special_tokens=True)
        all_tokens.append(tokens.cpu())
        all_texts.extend(texts)
        remaining -= current_batch

    stacked_tokens = torch.cat(all_tokens, dim=0)
    sampler.metrics.record_entropy(stacked_tokens)
    sampler.metrics.record_generative_perplexity(
        all_texts,
        max_length=config.model.length,
        device=sampler.device,
    )
    return {
        "sampler": name,
        "num_samples": total_samples,
        "perplexity": float(sampler.metrics.gen_ppl.compute().item()),
        "entropy": float(sampler.metrics.sample_entropy.compute().item()),
    }

for sampler_name, sampler in [
    ("ode", ode_sampler),
    ("sde", sde_sampler),
    ("marginally_exact", marginally_exact_sampler),
]:
    print(evaluate_sampler(sampler_name, sampler))
